In [11]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import sqlite3
import json
import requests
import streamlit as st

In [7]:
# Initialization

load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if not google_api_key:
    raise ValueError("GOOGLE_API_KEY environment variable is not set.")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY environment variable is not set.")

openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

MODEL_groq = "llama-3.3-70b-versatile" # groq
MODEL_gemini ="gemini-2.5-flash"  # gemini

client = groq
MODEL = MODEL_groq


In [8]:
def log_study_session(subject, duration):
    conn = sqlite3.connect("workspace.db")
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS study_logs (subject TEXT, duration INTEGER)")
    cursor.execute("INSERT INTO study_logs VALUES (?, ?)", (subject, duration))
    conn.commit()
    conn.close()
    return f"Successfully logged {duration} minutes of {subject}."

def get_focus_image(subject):
    return f"https://image.pollinations.ai/prompt/cyberpunk_style_icon_for_{subject}_productivity"


In [9]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "log_study_session",
            "description": "Logs a study or coding session duration and subject to the database",
            "parameters": {
                "type": "object",
                "properties": {
                    "subject": {"type": "string", "description": "The topic studied"},
                    "duration": {"type": "integer", "description": "Minutes spent studying"}
                },
                "required": ["subject", "duration"]
            }
        }
    }
]

In [10]:
st.set_page_config(page_title="SentinAI Workspace", layout="wide")
st.title("🛡️ SentinAI Workspace Controller")

# Session State for Chat History
if "messages" not in st.session_state:
    st.session_state.messages = [{"role": "system", "content": "You are a Workspace Assistant. Use tools to log sessions."}]

# User Input
if prompt := st.chat_input("What did you work on today?"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    
    # 1. First LLM Call
    response = client.chat.completions.create(
        model=MODEL,
        messages=st.session_state.messages,
        tools=tools,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    
    # 2. Handle Tool Calls (The "While" loop equivalent)
    if response_message.tool_calls:
        for tool_call in response_message.tool_calls:
            if tool_call.function.name == "log_study_session":
                args = json.loads(tool_call.function.arguments)
                result = log_study_session(args['subject'], args['duration'])
                
                # Add Tool Result to History
                st.session_state.messages.append(response_message)
                st.session_state.messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": "log_study_session",
                    "content": result
                })
        
        # 3. Second LLM Call (To get final natural language answer)
        final_response = client.chat.completions.create(
            model=MODEL,
            messages=st.session_state.messages
        )
        assistant_text = final_response.choices[0].message.content
    else:
        assistant_text = response_message.content

    st.session_state.messages.append({"role": "assistant", "content": assistant_text})

# --- RENDER UI ---
for msg in st.session_state.messages:
    if msg["role"] in ["user", "assistant"]:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])
            # If assistant mentions a subject, show the "Multi-modal" image
            if msg["role"] == "assistant" and "logged" in msg["content"].lower():
                st.image(get_focus_image("coding"), width=300)

2026-04-10 02:36:53.919 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 02:36:53.922 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 02:36:53.922 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 02:36:53.922 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 02:36:53.928 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 02:36:53.932 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 02:36:53.934 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 02:36:53.938 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar